# Brain — Multi-Agent System
**Google Colab notebook** — clone the repo, install deps, mount Drive for persistence, run the brain.

Recommended runtime: **L4 GPU** (Colab Pro) or **T4** (free tier).

If you want local Ollama models instead of Gemini, jump to the **Ollama (optional)** section.

## 1 — Mount Google Drive (data persistence)
ChromaDB and the episode SQLite database will be stored on Drive so they survive session restarts.

In [1]:
from google.colab import drive
drive.mount('/content/drive')

import os, pathlib

# All persistent data goes here — survives session restarts
DRIVE_DATA = '/content/drive/MyDrive/Brain/data'
pathlib.Path(DRIVE_DATA).mkdir(parents=True, exist_ok=True)

# Symlink so the code always finds data/ in the repo root
REPO_DATA = '/content/Brain/data'
if os.path.islink(REPO_DATA):
    os.remove(REPO_DATA)
# (symlink created after clone in step 3)
print(f'Drive mounted. Data will persist at: {DRIVE_DATA}')

Mounted at /content/drive
Drive mounted. Data will persist at: /content/drive/MyDrive/Brain/data


## 2 — Clone the repo

In [2]:
import os

REPO_URL = 'https://github.com/Seydifa/Brain.git'
REPO_DIR = '/content/Brain'

if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
else:
    !git -C {REPO_DIR} pull

%cd {REPO_DIR}
print('Working directory:', os.getcwd())

Cloning into '/content/Brain'...
remote: Enumerating objects: 32, done.
remote: Counting objects: 100% (32/32), done.
remote: Compressing objects: 100% (27/27), done.
remote: Total 32 (delta 2), reused 32 (delta 2), pack-reused 0 (from 0)
Receiving objects: 100% (32/32), 26.26 KiB | 26.26 MiB/s, done.
Resolving deltas: 100% (2/2), done.
/content/Brain
Working directory: /content/Brain


## 3 — Link Drive data directory

In [3]:
import os

REPO_DATA = '/content/Brain/data'
DRIVE_DATA = '/content/drive/MyDrive/Brain/data'

# Remove any existing data/ folder in the repo
if os.path.exists(REPO_DATA) and not os.path.islink(REPO_DATA):
    import shutil
    shutil.rmtree(REPO_DATA)
if os.path.islink(REPO_DATA):
    os.remove(REPO_DATA)

os.symlink(DRIVE_DATA, REPO_DATA)
print(f'data/ -> {DRIVE_DATA}')

data/ -> /content/drive/MyDrive/Brain/data


## 4 — Install dependencies

In [4]:
!pip install -q \
    langgraph \
    langgraph-checkpoint-sqlite \
    langchain-google-genai \
    langchain-chroma \
    langchain-ollama \
    ddgs \
    httpx \
    python-dotenv

print('Dependencies installed.')

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.5/66.5 kB 8.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.4/46.4 kB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.0/23.0 MB 81.4 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 150.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 163.4/163.4 kB 19.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 32.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 112.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.2/17.2 MB 131.9 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.1/72.1 kB 9.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 180.2/180.2 kB 21.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.0/69.0 kB 8.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━

## 5 — API key
Paste your Gemini API key below (get one free at https://aistudio.google.com/apikey).

In [5]:
import os
from google.colab import userdata

# Option A: store key in Colab Secrets (Colab sidebar > key icon) — recommended
try:
    os.environ['GOOGLE_API_KEY'] = userdata.get('GOOGLE_API_KEY')
    print('Key loaded from Colab Secrets.')
except Exception:
    # Option B: paste directly (less secure)
    os.environ['GOOGLE_API_KEY'] = 'YOUR_GEMINI_API_KEY_HERE'
    print('Key set inline.')

# Write .env so python-dotenv picks it up too
with open('/content/Brain/.env', 'w') as f:
    f.write(f"GOOGLE_API_KEY={os.environ['GOOGLE_API_KEY']}\n")

Key set inline.


## 6 — (Optional) Ollama with local models
Skip this section if you want to use Gemini API.  
On L4 GPU: `llama3.1:8b` responds in ~2s per call. On T4: ~5s.

In [ ]:
# Install prerequisites and Ollama
!apt-get install -y -q zstd
!curl -fsSL https://ollama.com/install.sh | sh

import subprocess, time, shutil, os

# Reload PATH so the newly installed binary is found
os.environ['PATH'] = '/usr/local/bin:' + os.environ.get('PATH', '')

ollama_bin = shutil.which('ollama') or '/usr/local/bin/ollama'
if not os.path.isfile(ollama_bin):
    raise RuntimeError(f'Ollama not found at {ollama_bin} — install may have failed above.')

subprocess.Popen([ollama_bin, 'serve'], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
time.sleep(3)

# Pull a model — must support tool calling for the ReAct search agent:
# 'llama3.2:3b'    → 2 GB VRAM, fast, full tool support  ← recommended
# 'llama3.1:8b'    → 8 GB VRAM, best quality, full tool support
# 'mistral:7b'     → 8 GB VRAM, strong reasoning, tool support
OLLAMA_MODEL = 'llama3.2:3b'
!ollama pull {OLLAMA_MODEL}
print(f'Model {OLLAMA_MODEL} ready.')

Reading package lists...
Building dependency tree...
Reading state information...
The following NEW packages will be installed:
  zstd
0 upgraded, 1 newly installed, 0 to remove and 42 not upgraded.
Need to get 603 kB of archives.
After this operation, 1,695 kB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu jammy/main amd64 zstd amd64 1.4.8+dfsg-3build1 [603 kB]
Fetched 603 kB in 2s (333 kB/s)
Selecting previously unselected package zstd.
(Reading database ... 122354 files and directories currently installed.)
Preparing to unpack .../zstd_1.4.8+dfsg-3build1_amd64.deb ...
Unpacking zstd (1.4.8+dfsg-3build1) ...
Setting up zstd (1.4.8+dfsg-3build1) ...
Processing triggers for man-db (2.10.2-1) ...
>>> Cleaning up old version at /usr/local/lib/ollama
>>> Installing ollama to /usr/local
>>> Downloading ollama-linux-amd64.tar.zst
######################################################################## 100.0%
>>> Creating ollama user...
>>> Adding ollama user t

In [6]:
# Patch all agent files to use Ollama instead of Gemini
import re, pathlib, os, subprocess, shutil

# Resolve repo root
REPO_DIR = '/content/Brain' if os.path.isdir('/content/Brain') else os.getcwd()
_p = pathlib.Path(REPO_DIR)
while not (_p / 'core' / 'graph.py').exists() and _p.parent != _p:
    _p = _p.parent
REPO_DIR = str(_p)
os.chdir(REPO_DIR)
print(f'Repo root: {REPO_DIR}')

# Restore original files from git before patching (idempotent)
subprocess.run(['git', 'checkout', '--', '.'], cwd=REPO_DIR)
print('Restored original files from git.')

OLLAMA_MODEL = 'gemma3:4b'  # match what you pulled above

files = [
    "memory/store.py", "memory/agent.py",
    "agents/qa_agent.py", "agents/orchestrator.py",
    "agents/search_agent.py", "agents/goal_evaluator.py",
    "agents/search_validator.py",
]

for path in files:
    p = pathlib.Path(REPO_DIR) / path
    src = p.read_text()

    # --- Store.py: has compound import (ChatModel + Embeddings) ---
    # Do the compound replacement FIRST so the standalone regex doesn't mangle it
    src = src.replace(
        "from langchain_google_genai import ChatGoogleGenerativeAI, GoogleGenerativeAIEmbeddings",
        "from langchain_ollama import ChatOllama, OllamaEmbeddings"
    )
    # The above also handles store.py's embedding instantiation below
    src = src.replace(
        'GoogleGenerativeAIEmbeddings(model="models/gemini-embedding-001")',
        'OllamaEmbeddings(model="nomic-embed-text")'
    )

    # --- All files: replace standalone ChatGoogleGenerativeAI import ---
    src = src.replace(
        "from langchain_google_genai import ChatGoogleGenerativeAI",
        "from langchain_ollama import ChatOllama"
    )
    # Replace model instantiation
    src = re.sub(
        r'ChatGoogleGenerativeAI\(model="[^"]+"',
        f'ChatOllama(model="{OLLAMA_MODEL}"',
        src
    )

    p.write_text(src)
    print(f"patched {path}")

# Pull the embedding model
ollama_bin = shutil.which('ollama') or '/usr/local/bin/ollama'
subprocess.run([ollama_bin, 'pull', 'nomic-embed-text'])
print('Ollama patch complete.')

Repo root: /content/Brain
Restored original files from git.
patched memory/store.py
patched memory/agent.py
patched agents/qa_agent.py
patched agents/orchestrator.py
patched agents/search_agent.py
patched agents/goal_evaluator.py
patched agents/search_validator.py
Ollama patch complete.


## 7 — Load the graph

In [7]:
import sys
sys.path.insert(0, '/content/Brain')

import uuid
from dotenv import load_dotenv
load_dotenv()

from core.graph import get_graph

graph = get_graph()
thread_id = str(uuid.uuid4())
config = {'configurable': {'thread_id': thread_id}}

EMPTY_STATE = {
    'goal': '',
    'messages': [],
    'response': '',
    'status': 'empty',
    'oriented_context': {},
    'reasoning_trace': [],
    'retry_count': 0,
    'search_valid': False,
    'search_feedback': '',
    'qa_draft': '',
    'qa_approved': False,
    'qa_feedback': '',
    'qa_attempts': 0,
    'needs_clarification': False,
    'clarification_questions': [],
}

print('Graph loaded. Ready to ask questions.')

Graph loaded. Ready to ask questions.


## 8 — Ask the Brain
Change `goal` and re-run this cell for each question.

In [8]:
import time

goal = "What caused World War 2?"  # <-- change this

state = {**EMPTY_STATE, 'goal': goal}

# Retry on rate limit (Gemini free tier)
for attempt in range(4):
    try:
        result = graph.invoke(state, config=config)
        break
    except Exception as e:
        if '429' in str(e) or 'RESOURCE_EXHAUSTED' in str(e):
            wait = 15 * (attempt + 1)
            print(f'Rate limit — retrying in {wait}s...')
            time.sleep(wait)
        else:
            raise

# Handle clarification request
if result.get('needs_clarification'):
    print('Brain needs clarification:')
    for q in result.get('clarification_questions', []):
        print(f'  - {q}')
else:
    print('=== BRAIN RESPONSE ===')
    print(result.get('response', '(no response)'))

ResponseError: registry.ollama.ai/library/gemma3:4b does not support tools (status code: 400)

## 9 — Debug: inspect reasoning trace

In [ ]:
ctx   = result.get('oriented_context', {})
trace = result.get('reasoning_trace', [])

print(f"Turn type  : {ctx.get('turn_type', '?')}")
print(f"Coverage   : {ctx.get('coverage', '?')}")
print(f"Confidence : {ctx.get('knowledge_confidence', 0):.2f}")
print(f"Episode    : {ctx.get('current_episode_id', '?')}")
print()
print('Reasoning trace:')
for i, step in enumerate(trace, 1):
    print(f'  {i}. {step}')

## 10 — View episode history

In [ ]:
from memory.episodes import get_recent
import json

episodes = get_recent(10)
for ep in episodes:
    flags = json.loads(ep.get('flags') or '[]')
    print(f"[{ep['id']}]")
    print(f"  Request  : {ep['user_request'][:80]}")
    print(f"  Type     : {ep['turn_type']}")
    print(f"  Flags    : {flags}")
    print(f"  Response : {str(ep.get('chosen_response', ''))[:100]}...")
    print()